In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    BaggingRegressor
)

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge, ElasticNet

from src.config import (
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\pc\Desktop\yzta-2026-datathon


In [2]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [3]:
def add_features(df):
    df = df.copy()

    if {"rem_yuzdesi", "derin_uyku_yuzdesi"}.issubset(df.columns):
        df["toplam_kaliteli_uyku_yuzdesi"] = (
            df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
        )

        df["rem_derin_uyku_carpim"] = (
            df["rem_yuzdesi"] * df["derin_uyku_yuzdesi"]
        )

    if {"gecelik_uyanma_sayisi", "uykuya_dalma_suresi_dk"}.issubset(df.columns):
        df["uyku_bolunme_yuku"] = (
            df["gecelik_uyanma_sayisi"] * df["uykuya_dalma_suresi_dk"]
        )

        df["uyku_verimsizlik_skoru"] = (
            df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
        )

    if {"stres_skoru", "gunluk_calisma_saati"}.issubset(df.columns):
        df["stres_calisma_yuku"] = (
            df["stres_skoru"] * df["gunluk_calisma_saati"]
        )

    if {"uyku_oncesi_ekran_suresi_dk", "uyku_oncesi_kafein_mg"}.issubset(df.columns):
        df["ekran_kafein_yuku"] = (
            df["uyku_oncesi_ekran_suresi_dk"] + df["uyku_oncesi_kafein_mg"]
        )

    if "gunluk_adim_sayisi" in df.columns:
        df["adim_sayisi_bin"] = df["gunluk_adim_sayisi"] / 1000

    if {"dinlenik_nabiz_bpm", "stres_skoru"}.issubset(df.columns):
        df["nabiz_stres_yuku"] = (
            df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
        )

    if "vucut_kitle_indeksi" in df.columns:
        df["bmi_kategori"] = pd.cut(
            df["vucut_kitle_indeksi"],
            bins=[0, 18.5, 25, 30, np.inf],
            labels=["zayif", "normal", "kilolu", "obez"]
        ).astype("object")

    if "gun_tipi" in df.columns:
        df["hafta_sonu_flag"] = (df["gun_tipi"] == "Hafta sonu").astype(int)

    if "ruh_sagligi_durumu" in df.columns:
        risk_map = {
            "Saglikli": 0,
            "Anksiyete": 1,
            "Depresyon": 2,
            "Anksiyete ve depresyon": 3,
        }

        df["ruh_sagligi_risk_skoru"] = df["ruh_sagligi_durumu"].map(risk_map)

    return df

In [4]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (56000, 24)
Test shape: (24000, 23)
Sample submission shape: (2, 2)


In [5]:
train_fe = add_features(train)
test_fe = add_features(test)

print("Train shape before FE:", train.shape)
print("Train shape after FE:", train_fe.shape)

print("Test shape before FE:", test.shape)
print("Test shape after FE:", test_fe.shape)

Train shape before FE: (56000, 24)
Train shape after FE: (56000, 35)
Test shape before FE: (24000, 23)
Test shape after FE: (24000, 34)


In [6]:
X = train_fe.drop(columns=[TARGET, ID_COL])
y = train_fe[TARGET]

X_test = test_fe.drop(columns=[ID_COL])
test_ids = test_fe[ID_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (56000, 33)
y shape: (56000,)
X_test shape: (24000, 33)


In [7]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))

Numeric feature count: 25
Categorical feature count: 8


In [8]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [9]:
def run_cv_model(model_name, model, X, y, X_test):
    print("=" * 80)
    print(f"Model: {model_name}")

    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), N_SPLITS))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        print(f"Fold {fold}")

        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        valid_pred = pipeline.predict(X_valid_fold)
        valid_pred = np.clip(valid_pred, 0, 10)

        fold_rmse = rmse(y_valid_fold, valid_pred)
        fold_scores.append(fold_rmse)

        oof_pred[valid_idx] = valid_pred

        test_pred = pipeline.predict(X_test)
        test_pred = np.clip(test_pred, 0, 10)

        test_pred_folds[:, fold - 1] = test_pred

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    mean_rmse = np.mean(fold_scores)
    std_rmse = np.std(fold_scores)

    print(f"{model_name} CV RMSE: {mean_rmse:.5f} ± {std_rmse:.5f}")

    return {
        "model": model_name,
        "cv_rmse_mean": mean_rmse,
        "cv_rmse_std": std_rmse,
        "fold_scores": fold_scores,
        "oof_pred": oof_pred,
        "test_pred": test_pred_folds.mean(axis=1)
    }

In [10]:
extra_regression_models = {
    "hist_gradient_boosting": HistGradientBoostingRegressor(
        max_iter=1000,
        learning_rate=0.03,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=RANDOM_STATE
    ),

    "gradient_boosting_tuned": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.9,
        random_state=RANDOM_STATE
    ),

    "extra_trees_tuned": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "random_forest_tuned": RandomForestRegressor(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "ridge": Ridge(
        alpha=10.0
    ),

    "elasticnet": ElasticNet(
        alpha=0.01,
        l1_ratio=0.2,
        max_iter=10000,
        random_state=RANDOM_STATE
    ),

    "adaboost_tree": AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
        n_estimators=500,
        learning_rate=0.03,
        random_state=RANDOM_STATE
    ),

    "bagging_tree": BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

In [11]:
all_results = []
all_oof_predictions = {}
all_test_predictions = {}

for model_name, model in extra_regression_models.items():
    result = run_cv_model(model_name, model, X, y, X_test)

    all_results.append({
        "model": result["model"],
        "cv_rmse_mean": result["cv_rmse_mean"],
        "cv_rmse_std": result["cv_rmse_std"],
        "fold_scores": result["fold_scores"]
    })

    all_oof_predictions[model_name] = result["oof_pred"]
    all_test_predictions[model_name] = result["test_pred"]

Model: hist_gradient_boosting
Fold 1
Fold 1 RMSE: 1.22861
Fold 2
Fold 2 RMSE: 1.22639
Fold 3
Fold 3 RMSE: 1.21435
Fold 4
Fold 4 RMSE: 1.22155
Fold 5
Fold 5 RMSE: 1.24287
hist_gradient_boosting CV RMSE: 1.22675 ± 0.00943
Model: gradient_boosting_tuned
Fold 1
Fold 1 RMSE: 1.23326
Fold 2
Fold 2 RMSE: 1.23015
Fold 3
Fold 3 RMSE: 1.21714
Fold 4
Fold 4 RMSE: 1.22635
Fold 5
Fold 5 RMSE: 1.24654
gradient_boosting_tuned CV RMSE: 1.23069 ± 0.00960
Model: extra_trees_tuned
Fold 1
Fold 1 RMSE: 1.26331
Fold 2
Fold 2 RMSE: 1.25999
Fold 3
Fold 3 RMSE: 1.25041
Fold 4
Fold 4 RMSE: 1.25751
Fold 5
Fold 5 RMSE: 1.27770
extra_trees_tuned CV RMSE: 1.26178 ± 0.00902
Model: random_forest_tuned
Fold 1
Fold 1 RMSE: 1.27585
Fold 2
Fold 2 RMSE: 1.26983
Fold 3
Fold 3 RMSE: 1.25830
Fold 4
Fold 4 RMSE: 1.27544
Fold 5
Fold 5 RMSE: 1.29289
random_forest_tuned CV RMSE: 1.27446 ± 0.01118
Model: ridge
Fold 1
Fold 1 RMSE: 1.25602
Fold 2
Fold 2 RMSE: 1.25059
Fold 3
Fold 3 RMSE: 1.23848
Fold 4
Fold 4 RMSE: 1.24440
Fold 5
Fo

In [12]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("cv_rmse_mean").reset_index(drop=True)

results_df[["model", "cv_rmse_mean", "cv_rmse_std"]]

,model,cv_rmse_mean,cv_rmse_std
0,hist_gradient_boosting,1.226754,0.009428
1,gradient_boosting_tuned,1.230688,0.009598
2,ridge,1.252480,0.011789
3,elasticnet,1.254471,0.011441
4,extra_trees_tuned,1.261781,0.009017
5,random_forest_tuned,1.274461,0.011181
6,bagging_tree,1.291849,0.010871
7,adaboost_tree,1.392966,0.005755


In [13]:
best_model_name = results_df.loc[0, "model"]
best_test_pred = all_test_predictions[best_model_name]

best_test_pred = np.clip(best_test_pred, 0, 10)

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_test_pred
})

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

submission_path = SUBMISSION_DIR / f"submission_{best_model_name}.csv"

submission.to_csv(submission_path, index=False)

print("Best model:", best_model_name)
print("Saved submission:", submission_path)
print("Submission shape:", submission.shape)
print("Submission columns:", submission.columns.tolist())

display(submission.head())

Best model: hist_gradient_boosting
Saved submission: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_hist_gradient_boosting.csv
Submission shape: (24000, 2)
Submission columns: ['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.868627
1,2,6.304210
2,3,3.042485
3,4,7.279240
4,5,3.674512


In [14]:
PREDICTION_DIR = PROJECT_ROOT / "predictions" / "extra_regressors"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

for model_name in all_oof_predictions.keys():
    oof_df = pd.DataFrame({
        ID_COL: train[ID_COL],
        "y_true": y,
        "oof_pred": np.clip(all_oof_predictions[model_name], 0, 10)
    })

    test_df = pd.DataFrame({
        ID_COL: test_ids,
        "test_pred": np.clip(all_test_predictions[model_name], 0, 10)
    })

    oof_path = PREDICTION_DIR / f"{model_name}_oof.csv"
    test_path = PREDICTION_DIR / f"{model_name}_test.csv"

    oof_df.to_csv(oof_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Saved:", oof_path)
    print("Saved:", test_path)

Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\hist_gradient_boosting_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\hist_gradient_boosting_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\gradient_boosting_tuned_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\gradient_boosting_tuned_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\extra_trees_tuned_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\extra_trees_tuned_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\random_forest_tuned_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\random_forest_tuned_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressors\ridge_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\extra_regressor

In [15]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / "extra_regressors_results.csv"

results_df.to_csv(results_path, index=False)

print("Saved results:", results_path)
display(results_df)

Saved results: c:\Users\pc\Desktop\yzta-2026-datathon\results\extra_regressors_results.csv


,model,cv_rmse_mean,cv_rmse_std,fold_scores
0,hist_gradient_boosting,1.226754,0.009428,"[1.2286077299932137, 1.2263898380942657, 1.214..."
1,gradient_boosting_tuned,1.230688,0.009598,"[1.2332608533082297, 1.2301460826953032, 1.217..."
2,ridge,1.252480,0.011789,"[1.2560207287435536, 1.250585409315132, 1.2384..."
3,elasticnet,1.254471,0.011441,"[1.2581535441678222, 1.252012317774789, 1.2404..."
4,extra_trees_tuned,1.261781,0.009017,"[1.263306226067136, 1.259992461498294, 1.25040..."
5,random_forest_tuned,1.274461,0.011181,"[1.2758499174124585, 1.269826572653033, 1.2582..."
6,bagging_tree,1.291849,0.010871,"[1.2930959585202664, 1.2879154360873415, 1.275..."
7,adaboost_tree,1.392966,0.005755,"[1.395741702945463, 1.3922775152371312, 1.3841..."
